# Scraping Data from API and making Data Frame and doing Text Preprocessing


## Importing Libraries


In [1]:
import requests
import pandas as pd
import string

## Scraping Data and Making Data Frame


### URL for API request of Movies


In [2]:
url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=1"


### Fetching data from API and converting to JSON and take a look at data


In [3]:
response = requests.get(url)
data = response.json()

In [4]:
data.keys()

dict_keys(['page', 'results', 'total_pages', 'total_results'])

In [5]:
data['results'][0]['id']

278

In [6]:
for key, value in data.items():
    print(f"{key}: {value}")

page: 1
results: [{'adult': False, 'backdrop_path': '/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg', 'genre_ids': [18, 80], 'id': 278, 'original_language': 'en', 'original_title': 'The Shawshank Redemption', 'overview': 'Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.', 'popularity': 29.7348, 'poster_path': '/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg', 'release_date': '1994-09-23', 'title': 'The Shawshank Redemption', 'video': False, 'vote_average': 8.715, 'vote_count': 29633}, {'adult': False, 'backdrop_path': '/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg', 'genre_ids': [18, 80], 'id': 238, 'original_language': 'en', 'original_title': 'The Godfather', 'overview': 'Spanning the yea

In [7]:
data['results'][0]['title']

'The Shawshank Redemption'

In [8]:
data['results'][0]['overview']

'Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.'

In [9]:
data['results'][0]['genre_ids']

[18, 80]

In [10]:
type(data['results'])

list

In [11]:
len(data['results'])

20

In [12]:
for i in data['results']:
    print(i['title'])

The Shawshank Redemption
The Godfather
The Godfather Part II
Schindler's List
12 Angry Men
Spirited Away
The Dark Knight
Dilwale Dulhania Le Jayenge
The Green Mile
Parasite
The Lord of the Rings: The Return of the King
Pulp Fiction
Your Name.
The Good, the Bad and the Ugly
Interstellar
Forrest Gump
GoodFellas
Seven Samurai
Grave of the Fireflies
Life Is Beautiful


### Fetching Genre Data


In [13]:
genres_url = "https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US"
genres_response = requests.get(genres_url)
genres_data = genres_response.json()

### Making Data Frame for Movies with Genre Names


In [14]:
df_genres = pd.DataFrame(genres_data['genres'], columns=['id', 'name'])

In [15]:
df_genres

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime
5,99,Documentary
6,18,Drama
7,10751,Family
8,14,Fantasy
9,36,History


### Movies with Genre Names Data Frame


In [16]:
df_movies = pd.DataFrame()

In [17]:
df_movies['Title'] = []
df_movies['Overview'] = []
df_movies['Genre'] = []

In [18]:
df_movies

,Title,Overview,Genre


In [19]:
df_movies['Title'] = "mani"

In [20]:
df_movies['Title']

,Title


In [21]:
genre_list = []

In [22]:
data['results'][0]['genre_ids']

[18, 80]

In [23]:
df_genres

,id,name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime
5,99,Documentary
6,18,Drama
7,10751,Family
8,14,Fantasy
9,36,History


In [24]:
for i in data['results'][0]['genre_ids']:
    genre_list.append(df_genres[df_genres['id']==i]['name'].values[0])

In [25]:
genre_list

['Drama', 'Crime']

In [26]:
','.join(genre_list)

'Drama,Crime'

### Fetching Top Rated Movies Data from API (500 Pages) with Genre Names


In [27]:
# movies with genre names list
movie_list = []
genre_list = []

# loop to iterate through 500 pages
for page in range(1,501):

    # url to fetch top rated movies
    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={page}"
    # getting response from the API
    response = requests.get(url)
    # covering response to json
    data = response.json()

    # loop to iterate throough each movie in results
    for movie in data['results']:

        # loop to get Genre Names from Genre IDs and adding to list
        for genre_id in movie['genre_ids']:
            # extracting genre name from genre id
            genre_name = df_genres[df_genres['id']==genre_id]['name'].values[0]

            # checking if genre name is already in the list or not
            if genre_name not in genre_list:
                genre_list.append(genre_name)

        # adding movie details to movie list
        movie_list.append({
            'Title': movie['title'],
            'Overview': movie['overview'],
            'Genres': ', '.join(genre_list)
        })

# creating dataframe from movie list
df_movies = pd.DataFrame(movie_list)

In [28]:
df_movies

,Title,Overview,Genres
0,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"Drama, Crime"
1,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama, Crime"
2,The Godfather Part II,In the continuing saga of the Corleone crime f...,"Drama, Crime"
3,Schindler's List,The true story of how businessman Oskar Schind...,"Drama, Crime, History, War"
4,12 Angry Men,The defense and the prosecution have rested an...,"Drama, Crime, History, War"
...,...,...,...
9995,The Farm,The young couple Nora and Alec are on their wa...,"Drama, Crime, History, War, Animation, Family,..."
9996,Silent House,Sarah returns with her father and uncle to fix...,"Drama, Crime, History, War, Animation, Family,..."
9997,The Next Karate Kid,"Mr. Miyagi decides to take Julie, a troubled t...","Drama, Crime, History, War, Animation, Family,..."
9998,Bachelorette,Three friends are asked to be bridesmaids at a...,"Drama, Crime, History, War, Animation, Family,..."


In [29]:
df_movies.shape

(10000, 3)

## Text Data Preprocessing


### Lowercase Conversion


In [30]:
df_movies['Overview'] = df_movies['Overview'].str.lower()

In [31]:
df_movies['Genres'] = df_movies['Genres'].str.lower()

In [32]:
df_movies['Title'] = df_movies['Title'].str.lower()

In [33]:
df_movies

,Title,Overview,Genres
0,the shawshank redemption,imprisoned in the 1940s for the double murder ...,"drama, crime"
1,the godfather,"spanning the years 1945 to 1955, a chronicle o...","drama, crime"
2,the godfather part ii,in the continuing saga of the corleone crime f...,"drama, crime"
3,schindler's list,the true story of how businessman oskar schind...,"drama, crime, history, war"
4,12 angry men,the defense and the prosecution have rested an...,"drama, crime, history, war"
...,...,...,...
9995,the farm,the young couple nora and alec are on their wa...,"drama, crime, history, war, animation, family,..."
9996,silent house,sarah returns with her father and uncle to fix...,"drama, crime, history, war, animation, family,..."
9997,the next karate kid,"mr. miyagi decides to take julie, a troubled t...","drama, crime, history, war, animation, family,..."
9998,bachelorette,three friends are asked to be bridesmaids at a...,"drama, crime, history, war, animation, family,..."


In [34]:
df_movies

,Title,Overview,Genres
0,the shawshank redemption,imprisoned in the 1940s for the double murder ...,"drama, crime"
1,the godfather,"spanning the years 1945 to 1955, a chronicle o...","drama, crime"
2,the godfather part ii,in the continuing saga of the corleone crime f...,"drama, crime"
3,schindler's list,the true story of how businessman oskar schind...,"drama, crime, history, war"
4,12 angry men,the defense and the prosecution have rested an...,"drama, crime, history, war"
...,...,...,...
9995,the farm,the young couple nora and alec are on their wa...,"drama, crime, history, war, animation, family,..."
9996,silent house,sarah returns with her father and uncle to fix...,"drama, crime, history, war, animation, family,..."
9997,the next karate kid,"mr. miyagi decides to take julie, a troubled t...","drama, crime, history, war, animation, family,..."
9998,bachelorette,three friends are asked to be bridesmaids at a...,"drama, crime, history, war, animation, family,..."


### Checking for HTML Tags in 'Overview' Column


In [35]:
# Check for HTML tags in Overview column
import re
html_pattern = r'<[^>]+>'
df_movies['has_html'] = df_movies['Overview'].str.contains(html_pattern, regex=True, na=False)
print(f"Rows with HTML tags in Overview: {df_movies['has_html'].sum()}")


Rows with HTML tags in Overview: 0


In [36]:
df_movies.drop(columns=['has_html'], inplace=True)

In [37]:
df_movies

,Title,Overview,Genres
0,the shawshank redemption,imprisoned in the 1940s for the double murder ...,"drama, crime"
1,the godfather,"spanning the years 1945 to 1955, a chronicle o...","drama, crime"
2,the godfather part ii,in the continuing saga of the corleone crime f...,"drama, crime"
3,schindler's list,the true story of how businessman oskar schind...,"drama, crime, history, war"
4,12 angry men,the defense and the prosecution have rested an...,"drama, crime, history, war"
...,...,...,...
9995,the farm,the young couple nora and alec are on their wa...,"drama, crime, history, war, animation, family,..."
9996,silent house,sarah returns with her father and uncle to fix...,"drama, crime, history, war, animation, family,..."
9997,the next karate kid,"mr. miyagi decides to take julie, a troubled t...","drama, crime, history, war, animation, family,..."
9998,bachelorette,three friends are asked to be bridesmaids at a...,"drama, crime, history, war, animation, family,..."


### Search url in Text


In [38]:
# Check for URLs in Overview column
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

df_movies['has_url'] = df_movies['Overview'].str.contains(
    url_pattern, regex=True, na=False
)


In [39]:
df_movies['has_url'].sum()

np.int64(0)

In [40]:
df_movies.drop(columns=['has_url'], inplace=True)

In [41]:
df_movies

,Title,Overview,Genres
0,the shawshank redemption,imprisoned in the 1940s for the double murder ...,"drama, crime"
1,the godfather,"spanning the years 1945 to 1955, a chronicle o...","drama, crime"
2,the godfather part ii,in the continuing saga of the corleone crime f...,"drama, crime"
3,schindler's list,the true story of how businessman oskar schind...,"drama, crime, history, war"
4,12 angry men,the defense and the prosecution have rested an...,"drama, crime, history, war"
...,...,...,...
9995,the farm,the young couple nora and alec are on their wa...,"drama, crime, history, war, animation, family,..."
9996,silent house,sarah returns with her father and uncle to fix...,"drama, crime, history, war, animation, family,..."
9997,the next karate kid,"mr. miyagi decides to take julie, a troubled t...","drama, crime, history, war, animation, family,..."
9998,bachelorette,three friends are asked to be bridesmaids at a...,"drama, crime, history, war, animation, family,..."


### Removing Punctuation

In [42]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [43]:
def remove_punct(text):
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    return text

In [44]:
df_movies['Overview'] = df_movies['Overview'].apply(remove_punct)

In [45]:
df_movies['Genres'] = df_movies['Genres'].apply(remove_punct)

In [46]:
df_movies['Title'] = df_movies['Title'].apply(remove_punct)

In [47]:
df_movies.iloc[0]['Title']

'the shawshank redemption'

In [48]:
df_movies.iloc[0]['Overview']

'imprisoned in the 1940s for the double murder of his wife and her lover upstanding banker andy dufresne begins a new life at the shawshank prison where he puts his accounting skills to work for an amoral warden during his long stretch in prison dufresne comes to be admired by the other inmates  including an older prisoner named red  for his integrity and unquenchable sense of hope'

### Removing Stop Words

In [65]:
import nltk

In [66]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [67]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [68]:
def remove_stopwords(text):
    cleaned_text = [word for word in text.split() if word not in stop_words]
    return ' '.join(cleaned_text)

In [69]:
text = "The cat is sitting on the mat"
remove_stopwords(text)
# Output: "cat sitting mat"

'The cat sitting mat'

In [70]:
df_movies['Overview'] = df_movies['Overview'].apply(remove_stopwords)

In [71]:
df_movies['Genres'] = df_movies['Genres'].apply(remove_stopwords)

In [72]:
df_movies['Title'] = df_movies['Title'].apply(remove_stopwords)

In [73]:
df_movies

,Title,Overview,Genres
0,shawshank redemption,imprisoned 1940s double murder wife lover upst...,drama crime
1,godfather,spanning years 1945 1955 chronicle fictional i...,drama crime
2,godfather part ii,continuing saga corleone crime family young vi...,drama crime
3,schindlers list,true story businessman oskar schindler saved t...,drama crime history war
4,12 angry men,defense prosecution rested jury filing jury ro...,drama crime history war
...,...,...,...
9995,farm,young couple nora alec way back long road trip...,drama crime history war animation family fanta...
9996,silent house,sarah returns father uncle fix familys longtim...,drama crime history war animation family fanta...
9997,next karate kid,mr miyagi decides take julie troubled teenager...,drama crime history war animation family fanta...
9998,bachelorette,three friends asked bridesmaids wedding woman ...,drama crime history war animation family fanta...


### Removing Stemming

In [84]:
from nltk.stem import PorterStemmer

In [85]:
ps = PorterStemmer()
def stemming(text):
    return " ".join([ps.stem(word) for word in text.split()])

In [87]:
sentence = "walk walking walked walks"
stem =  stemming(sentence)
print(stem)

walk walk walk walk


In [88]:
df_movies['Title'] = df_movies['Title'].apply(stemming)
df_movies['Overview'] = df_movies['Overview'].apply(stemming)
df_movies['Genres'] = df_movies['Genres'].apply(stemming)

### Lemmatization

In [93]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [94]:
lemmatizer = WordNetLemmatizer()

In [95]:
def lemmatize_text(text):
    words = text.split()
    lammatized = [lemmatizer.lemmatize(word,pos='v') for word in words]
    return ' '.join(lammatized)

In [96]:
df_movies['Title'] = df_movies['Title'].apply(lemmatize_text)
df_movies['Overview'] = df_movies['Overview'].apply(lemmatize_text)
df_movies['Genres'] = df_movies['Genres'].apply(lemmatize_text)

### Tokenization

In [97]:
from nltk.tokenize import word_tokenize

In [98]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [99]:
def apply_token(text):
    return word_tokenize(text)

In [100]:
df_movies['Title_Token'] = df_movies['Title'].apply(apply_token)
df_movies['Overview_Token'] = df_movies['Overview'].apply(apply_token)
df_movies['Genres_Token'] = df_movies['Genres'].apply(apply_token)